# Huấn luyện 2 mô hình YOLOv8 cho nhận diện biển số xe (Google Colab)

Sổ tay này làm đúng như mục 3.2.1 của tiểu luận: **2 mô hình YOLOv8**.

| | Mô hình 1: vùng biển số | Mô hình 2: ký tự |
|---|---|---|
| Số lớp (`nc`) | 1 (`plate`) | 36 (số 0-9 và chữ A-Z) |
| Kích thước ảnh | 640 | 640 |
| Batch | 24 | 24 |
| Số epoch | 100 | 50 |
| File kết quả | `plate.pt` | `char.pt` |

**File `.yaml` (trong báo cáo gọi là `coco.yaml`) dùng để làm gì?** Đây là file cấu hình dữ liệu cho bước huấn luyện. Nó cho YOLO biết
(1) ảnh train / val / test nằm ở thư mục nào (mỗi ảnh có 1 file nhãn `.txt` cùng tên),
(2) có bao nhiêu lớp (`nc`), và (3) tên từng lớp theo thứ tự 0, 1, 2... (số thứ tự này chính là số lớp ghi trong file nhãn).
Huấn luyện xong, tên lớp được lưu luôn vào `plate.pt` / `char.pt`, nên chương trình nhận diện không cần đọc lại file yaml.
Tên file `coco.yaml` là file mẫu của Ultralytics (bộ dữ liệu COCO, 80 lớp) mà báo cáo sửa lại; ở đây vẫn đặt tên `coco.yaml` như báo cáo: mỗi mô hình một file, nằm trong thư mục dữ liệu riêng của nó (Hình 3.3 cho mô hình biển số, Hình 3.8 cho mô hình ký tự).

**Cách dùng:** chọn *Runtime → Change runtime type → T4 GPU*, rồi chạy lần lượt từng ô từ trên xuống.

In [ ]:
# 1) Kiểm tra GPU
!nvidia-smi
import torch
print("Có GPU:", torch.cuda.is_available())

In [ ]:
# 2) Cài thư viện
!pip -q install ultralytics roboflow pyyaml

In [ ]:
# 3) Nối Google Drive để lưu kết quả (tránh mất khi Colab ngắt kết nối)
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = "/content/drive/MyDrive/bien_so_yolo"
os.makedirs(OUT, exist_ok=True)
print("Kết quả sẽ lưu tại:", OUT)

## Dữ liệu

Báo cáo dùng dữ liệu gán nhãn trên Roboflow: bộ ảnh biển số (khoảng 24 nghìn ảnh) và bộ ảnh ký tự (khoảng 4,6 nghìn ảnh).
Bạn cần điền **API key** và **tên workspace / project / version** của bộ dữ liệu bạn dùng (lấy ở trang Roboflow: *Versions → Download Dataset → YOLOv8 → show download code*).

- Nếu chưa có dữ liệu: vào Roboflow Universe, tìm "license plate" (vùng biển số) và bộ ký tự biển số Việt Nam, chọn *Fork* về tài khoản của bạn rồi lấy thông tin ở trên.
- Nếu tự gán nhãn (labelImg, Roboflow): mỗi ảnh một file `.txt` cùng tên, mỗi dòng `lop x_giua y_giua rong cao` (chuẩn hóa 0-1), xếp theo thư mục `train/images`, `train/labels`, `valid/...`, `test/...`.
- Bộ ký tự phải có **36 lớp**. Thứ tự lớp do bộ dữ liệu quy định, ô bên dưới giữ nguyên thứ tự đó, không tự đổi.

In [ ]:
# 4) Điền thông tin dữ liệu của bạn
ROBOFLOW_API_KEY = "DIEN_API_KEY_CUA_BAN"
PLATE_DATA = {"workspace": "TEN_WORKSPACE", "project": "TEN_PROJECT_VUNG_BIEN_SO", "version": 1}
CHAR_DATA  = {"workspace": "TEN_WORKSPACE", "project": "TEN_PROJECT_KY_TU", "version": 1}

In [ ]:
# 5) Hàm tải dữ liệu và tạo file yaml (vai trò của "coco.yaml" trong báo cáo)
import yaml
from roboflow import Roboflow

def tai_dataset(cfg):
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    ds = rf.workspace(cfg["workspace"]).project(cfg["project"]).version(cfg["version"]).download("yolov8")
    return ds.location

def tao_yaml(thu_muc, ten_file):
    """Đọc data.yaml của Roboflow (giữ nguyên số lớp + tên lớp), ghi lại file yaml mới với đường dẫn tuyệt đối."""
    goc = yaml.safe_load(open(os.path.join(thu_muc, "data.yaml"), encoding="utf-8"))
    names = goc["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    cfg = {"path": thu_muc, "train": "train/images"}
    cfg["val"] = "valid/images" if os.path.isdir(os.path.join(thu_muc, "valid", "images")) else "val/images"
    if os.path.isdir(os.path.join(thu_muc, "test", "images")):
        cfg["test"] = "test/images"
    cfg["nc"] = len(names)
    cfg["names"] = names
    out = os.path.join(thu_muc, ten_file)
    with open(out, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
    print(open(out, encoding="utf-8").read())
    return out

## Mô hình 1: phát hiện vùng chứa biển số (1 lớp, 100 epoch)

In [ ]:
# 6) Tải dữ liệu biển số + tạo plate.yaml
PLATE_DIR = tai_dataset(PLATE_DATA)
PLATE_YAML = tao_yaml(PLATE_DIR, "coco.yaml")

In [ ]:
# 7) Huấn luyện mô hình 1 (khoảng vài chục phút đến vài giờ tùy số ảnh)
from ultralytics import YOLO
model_plate = YOLO("yolov8n.pt")      # trọng số có sẵn; muốn chính xác hơn có thể dùng yolov8s.pt
model_plate.train(data=PLATE_YAML, imgsz=640, batch=24, epochs=100,
                  project=OUT, name="plate", exist_ok=True)

In [ ]:
# 8) Xem biểu đồ huấn luyện (giống Hình 3.5 trong báo cáo) và đánh giá
from IPython.display import Image, display
display(Image(f"{OUT}/plate/results.png"))
display(Image(f"{OUT}/plate/confusion_matrix.png"))

best_plate = YOLO(f"{OUT}/plate/weights/best.pt")
split_plate = "test" if "test" in yaml.safe_load(open(PLATE_YAML)) else "val"
m = best_plate.val(data=PLATE_YAML, split=split_plate)
print(f"Precision={m.box.mp:.3f}  Recall={m.box.mr:.3f}  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")

## Mô hình 2: nhận diện ký tự (36 lớp, 50 epoch)

In [ ]:
# 9) Tải dữ liệu ký tự + tạo char.yaml
CHAR_DIR = tai_dataset(CHAR_DATA)
CHAR_YAML = tao_yaml(CHAR_DIR, "coco.yaml")
n_lop = len(yaml.safe_load(open(CHAR_YAML))["names"])
print("Số lớp ký tự:", n_lop)
if n_lop != 36:
    print("CẢNH BÁO: báo cáo dùng 36 lớp (0-9 và A-Z). Hãy kiểm tra lại bộ dữ liệu ký tự.")

In [ ]:
# 10) Huấn luyện mô hình 2
model_char = YOLO("yolov8n.pt")
model_char.train(data=CHAR_YAML, imgsz=640, batch=24, epochs=50,
                 project=OUT, name="char", exist_ok=True)

In [ ]:
# 11) Biểu đồ huấn luyện và độ chính xác từng ký tự (giống Hình 3.10, 3.11)
display(Image(f"{OUT}/char/results.png"))
display(Image(f"{OUT}/char/confusion_matrix.png"))

best_char = YOLO(f"{OUT}/char/weights/best.pt")
split_char = "test" if "test" in yaml.safe_load(open(CHAR_YAML)) else "val"
m = best_char.val(data=CHAR_YAML, split=split_char)
print(f"Tổng: Precision={m.box.mp:.3f}  Recall={m.box.mr:.3f}  mAP50={m.box.map50:.3f}")
for i, ap in zip(m.box.ap_class_index, m.box.ap50):
    print(best_char.names[int(i)], round(float(ap), 3))

## Thử nhận diện kết hợp 2 mô hình (giống Hình 3.12, 3.13)

In [ ]:
# 12) Ảnh -> mô hình 1 cắt biển số -> mô hình 2 đọc ký tự -> ghép chuỗi
import cv2, glob, numpy as np
from google.colab.patches import cv2_imshow

def doc_bien_so(duong_dan):
    anh = cv2.imread(duong_dan)
    ket_qua = []
    for x1, y1, x2, y2 in best_plate(anh, conf=0.25, verbose=False)[0].boxes.xyxy.tolist():
        crop = anh[int(y1):int(y2), int(x1):int(x2)]          # cắt theo tọa độ xyxy
        r = best_char(crop, conf=0.25, verbose=False, agnostic_nms=True)[0]
        ds = [((b[0] + b[2]) / 2, (b[1] + b[3]) / 2, b[3] - b[1], best_char.names[int(c)])
              for b, c in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist())]
        if not ds:
            continue
        cao = float(np.median([d[2] for d in ds]))
        ys = [d[1] for d in ds]
        if max(ys) - min(ys) > 0.7 * cao:                      # biển 2 dòng: đọc dòng trên rồi dòng dưới
            giua = (max(ys) + min(ys)) / 2
            dong = [sorted([d for d in ds if d[1] < giua]), sorted([d for d in ds if d[1] >= giua])]
        else:
            dong = [sorted(ds)]
        ket_qua.append(" ".join("".join(str(d[3]).upper() for d in d1) for d1 in dong))
    return ket_qua, anh

thu = sorted(glob.glob(PLATE_DIR + "/test/images/*"))[:6] or sorted(glob.glob(PLATE_DIR + "/valid/images/*"))[:6]
for duong_dan in thu:
    kq, anh = doc_bien_so(duong_dan)
    print(os.path.basename(duong_dan), "->", kq)
    cv2_imshow(cv2.resize(anh, None, fx=0.5, fy=0.5))

## Lưu và tải trọng số về máy

Hai file `plate.pt` và `char.pt` được lưu vào Drive (thư mục `bien_so_yolo`) và tải về máy. Chép chúng vào cùng thư mục với `bien_so_yolo.py`, chương trình sẽ tự dùng.

Nếu Colab bị ngắt giữa chừng, chạy tiếp từ điểm dừng (bỏ dấu `#`):

In [ ]:
# Huấn luyện tiếp khi bị ngắt kết nối (bỏ dấu # nếu cần)
# YOLO(f"{OUT}/plate/weights/last.pt").train(resume=True)
# YOLO(f"{OUT}/char/weights/last.pt").train(resume=True)

# 13) Đổi tên và tải về
import shutil
shutil.copy(f"{OUT}/plate/weights/best.pt", f"{OUT}/plate.pt")
shutil.copy(f"{OUT}/char/weights/best.pt", f"{OUT}/char.pt")
from google.colab import files
files.download(f"{OUT}/plate.pt")
files.download(f"{OUT}/char.pt")